In [1]:
%cd /kaggle/working

!pip -q install -U transformers sentencepiece spacy scikit-learn pandas tqdm nltk pytorch-lightning
!python -m spacy download en_core_web_sm -q

!rm -rf /kaggle/working/AlignScore
!git clone https://github.com/yuh-zha/AlignScore.git /kaggle/working/AlignScore

# Patch 1: AdamW issue
MODEL_FILE = "/kaggle/working/AlignScore/src/alignscore/model.py"

with open(MODEL_FILE, "r", encoding="utf-8") as f:
    code = f.read()

code = code.replace(
    "from transformers import AdamW, get_linear_schedule_with_warmup, AutoConfig",
    "from torch.optim import AdamW\nfrom transformers import get_linear_schedule_with_warmup, AutoConfig"
)

with open(MODEL_FILE, "w", encoding="utf-8") as f:
    f.write(code)

# Patch 2: pytorch_lightning load_from_checkpoint issue
INFER_FILE = "/kaggle/working/AlignScore/src/alignscore/inference.py"

with open(INFER_FILE, "r", encoding="utf-8") as f:
    code = f.read()

code = code.replace(
    "self.model = BERTAlignModel(model=model).load_from_checkpoint(checkpoint_path=ckpt_path, strict=False).to(self.device)",
    "self.model = BERTAlignModel.load_from_checkpoint(checkpoint_path=ckpt_path, model=model, strict=False).to(self.device)"
)

with open(INFER_FILE, "w", encoding="utf-8") as f:
    f.write(code)

import sys
sys.path.insert(0, "/kaggle/working/AlignScore/src")

from alignscore.alignscore import AlignScore

print("AlignScore ready ✅")

/kaggle/working
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 99.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 92.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 101.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 44.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.0 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profilin

In [2]:
import os

CKPT_PATH = "/kaggle/working/AlignScore-base.ckpt"

if not os.path.exists(CKPT_PATH):
    !wget -q -O /kaggle/working/AlignScore-base.ckpt https://huggingface.co/yzha/AlignScore/resolve/main/AlignScore-base.ckpt

print("Checkpoint exists:", os.path.exists(CKPT_PATH))

Checkpoint exists: True


In [3]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

INPUT_CSV = "/kaggle/input/datasets/belalheshamfathy/oragnai/test_predictions _oragnail.csv"
OUTPUT_DIR = "/kaggle/working/alignscore_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 8
DEVICE = 0 if torch.cuda.is_available() else -1
THRESHOLDS = [0.30, 0.40, 0.50, 0.60, 0.70]

df = pd.read_csv(INPUT_CSV)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df = df.dropna(subset=["knowledge", "question", "answer", "label"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)

contexts = (
    "Knowledge:\n" + df["knowledge"].astype(str) +
    "\n\nQuestion:\n" + df["question"].astype(str)
).tolist()

claims = df["answer"].astype(str).tolist()
labels = df["label"].values

scorer = AlignScore(
    model="roberta-base",
    batch_size=BATCH_SIZE,
    device=DEVICE,
    ckpt_path=CKPT_PATH,
    evaluation_mode="nli_sp"
)

scores = []

for i in tqdm(range(0, len(contexts), BATCH_SIZE), desc="AlignScore scoring"):
    batch_scores = scorer.score(
        contexts=contexts[i:i+BATCH_SIZE],
        claims=claims[i:i+BATCH_SIZE]
    )
    scores.extend(batch_scores)

scores = np.array(scores, dtype=float)
df["alignscore"] = scores

results = []

for th in THRESHOLDS:
    preds = (scores >= th).astype(int)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary",
        pos_label=1,
        zero_division=0
    )

    cm = confusion_matrix(labels, preds)

    results.append({
        "method": "AlignScore",
        "threshold": th,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tn": cm[0, 0],
        "fp": cm[0, 1],
        "fn": cm[1, 0],
        "tp": cm[1, 1],
    })

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)

BEST_THRESHOLD = float(results_df.iloc[0]["threshold"])
final_preds = (scores >= BEST_THRESHOLD).astype(int)

df["alignscore_prediction"] = final_preds
df["alignscore_correct"] = df["alignscore_prediction"] == df["label"]

df.to_csv(f"{OUTPUT_DIR}/alignscore_predictions.csv", index=False)
results_df.to_csv(f"{OUTPUT_DIR}/alignscore_threshold_results.csv", index=False)

precision, recall, f1, _ = precision_recall_fscore_support(
    labels,
    final_preds,
    average="binary",
    pos_label=1,
    zero_division=0
)

final_summary = pd.DataFrame([{
    "method": "AlignScore",
    "best_threshold": BEST_THRESHOLD,
    "accuracy": accuracy_score(labels, final_preds),
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "num_samples": len(df)
}])

final_summary.to_csv(f"{OUTPUT_DIR}/alignscore_final_summary.csv", index=False)

print("\nThreshold Results:")
display(results_df)

print("\nBest Threshold:", BEST_THRESHOLD)

print("\nClassification Report:")
print(classification_report(
    labels,
    final_preds,
    target_names=["Hallucinated", "Supported"],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(labels, final_preds))

print("\nFinal Summary:")
display(final_summary)

print("\nSaved in:", OUTPUT_DIR)

Shape: (6000, 6)
Columns: ['knowledge', 'question', 'answer', 'label', 'prediction', 'correct']


Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint AlignScore-base.ckpt`


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...


Threshold Results:


,method,threshold,accuracy,precision,recall,f1,tn,fp,fn,tp
0,AlignScore,0.3,0.770833,0.722664,0.879000,0.793202,1988,1012,363,2637
1,AlignScore,0.4,0.770333,0.736581,0.841667,0.785625,2097,903,475,2525
2,AlignScore,0.5,0.760500,0.745677,0.790667,0.767513,2191,809,628,2372
3,AlignScore,0.6,0.743167,0.753916,0.722000,0.737613,2293,707,834,2166
4,AlignScore,0.7,0.714833,0.755653,0.635000,0.690092,2384,616,1095,1905



Best Threshold: 0.3

Classification Report:
              precision    recall  f1-score   support

Hallucinated       0.85      0.66      0.74      3000
   Supported       0.72      0.88      0.79      3000

    accuracy                           0.77      6000
   macro avg       0.78      0.77      0.77      6000
weighted avg       0.78      0.77      0.77      6000


Confusion Matrix:
[[1988 1012]
 [ 363 2637]]

Final Summary:


,method,best_threshold,accuracy,precision,recall,f1,num_samples
0,AlignScore,0.3,0.770833,0.722664,0.879,0.793202,6000



Saved in: /kaggle/working/alignscore_outputs
